In [2]:
from google.colab import files

uploaded = files.upload()

Saving base_analitica_calidad_aire.xlsx to base_analitica_calidad_aire.xlsx


In [3]:
import pandas as pd
import altair as alt

df = pd.read_excel("base_analitica_calidad_aire.xlsx", sheet_name="pendientes")

df_validas = df[df['R2'] >= 0.4].copy()

def clasificar(row):
    if row['pendiente'] < 0:
        return 'Mejora'
    elif row['pendiente'] > 0:
        return 'Empeora'
    else:
        return 'Sin cambio'

df_validas['Tendencia'] = df_validas.apply(clasificar, axis=1)

color_scale = alt.Scale(domain=['Mejora','Empeora','Sin cambio'],
                        range=['#4CAF50','#E53935','#9E9E9E'])

chart = alt.Chart(df_validas).mark_bar().encode(
    x=alt.X('pendiente:Q', title='Pendiente anual (µg/m³ por año)'),
    y=alt.Y('estacion:N', sort='x', title='Estaciones'),
    color=alt.Color('Tendencia:N', scale=color_scale, legend=alt.Legend(title="Tendencia")),
    tooltip=['region','estacion','pendiente','R2']
).properties(
    title='Pendiente anual de PM2.5 por estación (R² ≥ 0.4)'
)

chart.save('grafico_pendientes.html')
chart


ValueError: Worksheet named 'pendientes' not found

In [4]:
import pandas as pd
import altair as alt
from sklearn.linear_model import LinearRegression
import numpy as np

# Cargar datos
df = pd.read_excel("base_analitica_calidad_aire.xlsx")

# Filtrar filas con valores numéricos
df = df.dropna(subset=['anio','valor'])
df = df[df['valor'].apply(lambda x: isinstance(x,(int,float)))]

# Calcular pendiente y R² por estación
resultados = []
for estacion, grupo in df.groupby('estacion'):
    X = grupo['anio'].values.reshape(-1,1)
    y = grupo['valor'].values
    if len(grupo) >= 5:  # mínimo 5 años para calcular
        modelo = LinearRegression().fit(X,y)
        pendiente = modelo.coef_[0]
        R2 = modelo.score(X,y)
        resultados.append([grupo['region'].iloc[0], estacion, pendiente, R2])

pendientes = pd.DataFrame(resultados, columns=['region','estacion','pendiente','R2'])

# Filtrar estaciones con R² >= 0.4
df_validas = pendientes[pendientes['R2'] >= 0.4].copy()

# Clasificación
df_validas['Tendencia'] = df_validas['pendiente'].apply(
    lambda x: 'Mejora' if x < 0 else ('Empeora' if x > 0 else 'Sin cambio')
)

# Colores
color_scale = alt.Scale(domain=['Mejora','Empeora','Sin cambio'],
                        range=['#4CAF50','#E53935','#9E9E9E'])

# Gráfico
chart = alt.Chart(df_validas).mark_bar().encode(
    x=alt.X('pendiente:Q', title='Pendiente anual (µg/m³ por año)'),
    y=alt.Y('estacion:N', sort='x', title='Estaciones'),
    color=alt.Color('Tendencia:N', scale=color_scale, legend=alt.Legend(title="Tendencia")),
    tooltip=['region','estacion','pendiente','R2']
).properties(
    title='Pendiente anual de PM2.5 por estación (R² ≥ 0.4)'
)

chart.save('grafico_pendientes.html')
chart


alt.Chart(...)

In [5]:
import pandas as pd
import altair as alt
from sklearn.linear_model import LinearRegression
import numpy as np

df = pd.read_excel("base_analitica_calidad_aire.xlsx")

df = df.dropna(subset=['anio','valor'])
df = df[df['valor'].apply(lambda x: isinstance(x,(int,float)))]

resultados = []
for region, grupo_region in df.groupby('region'):
    X = grupo_region['anio'].values.reshape(-1,1)
    y = grupo_region['valor'].values
    if len(grupo_region) >= 5:
        modelo = LinearRegression().fit(X,y)
        pendiente = modelo.coef_[0]
        R2 = modelo.score(X,y)
        resultados.append([region, pendiente, R2])

pendientes_region = pd.DataFrame(resultados, columns=['region','pendiente','R2'])

df_validas = pendientes_region[pendientes_region['R2'] >= 0.4].copy()

df_validas['Tendencia'] = df_validas['pendiente'].apply(
    lambda x: 'Mejora' if x < 0 else ('Empeora' if x > 0 else 'Sin cambio')
)

color_scale = alt.Scale(domain=['Mejora','Empeora','Sin cambio'],
                        range=['#4CAF50','#E53935','#9E9E9E'])

chart = alt.Chart(df_validas).mark_bar().encode(
    x=alt.X('pendiente:Q', title='Pendiente anual (µg/m³ por año)'),
    y=alt.Y('region:N', sort='x', title='Regiones'),
    color=alt.Color('Tendencia:N', scale=color_scale, legend=alt.Legend(title="Tendencia")),
    tooltip=['region','pendiente','R2']
).properties(
    title='Pendiente anual de PM2.5 por región (R² ≥ 0.4)'
)

chart.save('grafico_pendientes_regiones.html')
chart


alt.Chart(...)

In [6]:
import pandas as pd
import altair as alt
from sklearn.linear_model import LinearRegression

# Cargar datos
df = pd.read_excel("base_analitica_calidad_aire.xlsx")

# Filtrar filas con valores numéricos
df = df.dropna(subset=['anio','valor'])
df = df[df['valor'].apply(lambda x: isinstance(x,(int,float)))]

# Calcular pendiente y R² por región
resultados = []
for region, grupo in df.groupby('region'):
    X = grupo['anio'].values.reshape(-1,1)
    y = grupo['valor'].values
    if len(grupo) >= 5:
        modelo = LinearRegression().fit(X,y)
        pendiente = modelo.coef_[0]
        R2 = modelo.score(X,y)
        resultados.append([region, pendiente, R2])

pendientes_region = pd.DataFrame(resultados, columns=['region','pendiente','R2'])

# Filtrar regiones con R² >= 0.4
df_validas = pendientes_region[pendientes_region['R2'] >= 0.4].copy()

# Clasificación
df_validas['Tendencia'] = df_validas['pendiente'].apply(
    lambda x: 'Mejora' if x < 0 else ('Empeora' if x > 0 else 'Sin cambio')
)

# Colores
color_scale = alt.Scale(domain=['Mejora','Empeora','Sin cambio'],
                        range=['#4CAF50','#E53935','#9E9E9E'])

# Gráfico de barras divergentes por región
chart = alt.Chart(df_validas).mark_bar().encode(
    x=alt.X('pendiente:Q', title='Pendiente anual (µg/m³ por año)'),
    y=alt.Y('region:N', sort='x', title='Regiones'),
    color=alt.Color('Tendencia:N', scale=color_scale, legend=alt.Legend(title="Tendencia")),
    tooltip=['region','pendiente','R2']
).properties(
    title='Pendiente anual de PM2.5 por región (R² ≥ 0.4)'
)

chart.save('grafico_pendientes_regiones.html')
chart


alt.Chart(...)

In [7]:
import pandas as pd
import altair as alt
from sklearn.linear_model import LinearRegression

df = pd.read_excel("base_analitica_calidad_aire.xlsx")

df = df.dropna(subset=['anio','valor'])
df = df[df['valor'].apply(lambda x: isinstance(x,(int,float)))]

# Promedio anual por región
df_region = df.groupby(['region','anio'], as_index=False)['valor'].mean()

resultados = []
for region, grupo in df_region.groupby('region'):
    X = grupo['anio'].values.reshape(-1,1)
    y = grupo['valor'].values
    if len(grupo) >= 5:
        modelo = LinearRegression().fit(X,y)
        pendiente = modelo.coef_[0]
        R2 = modelo.score(X,y)
        resultados.append([region, pendiente, R2])

pendientes_region = pd.DataFrame(resultados, columns=['region','pendiente','R2'])

df_validas = pendientes_region.copy()
df_validas['Tendencia'] = df_validas['pendiente'].apply(
    lambda x: 'Mejora' if x < 0 else ('Empeora' if x > 0 else 'Sin cambio')
)

color_scale = alt.Scale(domain=['Mejora','Empeora','Sin cambio'],
                        range=['#4CAF50','#E53935','#9E9E9E'])

chart = alt.Chart(df_validas).mark_bar().encode(
    x=alt.X('pendiente:Q', title='Pendiente anual (µg/m³ por año)'),
    y=alt.Y('region:N', sort='x', title='Regiones'),
    color=alt.Color('Tendencia:N', scale=color_scale, legend=alt.Legend(title="Tendencia")),
    tooltip=['region','pendiente','R2']
).properties(
    title='Pendiente anual de PM2.5 por región (R² ≥ 0.4)'
)

chart.save('grafico_pendientes_regiones.html')
chart


alt.Chart(...)

In [9]:

import pandas as pd
import altair as alt

df = pd.read_excel("base_analitica_calidad_aire.xlsx")

df_estaciones = df.groupby(['anio'])['estacion'].nunique().reset_index()

chart_acto1 = alt.Chart(df_estaciones).mark_bar(color='#4A90E2').encode(
    x=alt.X('anio:O', title='Año'),
    y=alt.Y('estacion:Q', title='Número de estaciones'),
    tooltip=['anio','estacion']
).properties(
    title='Expansión de la red de monitoreo (2000–2026)'
)

chart_acto1.save('visualizaciones/acto1_expansion_red.html')
chart_acto1


FileNotFoundError: [Errno 2] No such file or directory: 'visualizaciones/acto1_expansion_red.html'

In [10]:
import pandas as pd
import altair as alt
import os

os.makedirs("visualizacioness", exist_ok=True)

df = pd.read_excel("base_analitica_calidad_aire.xlsx")

df_estaciones = df.groupby(['anio'])['estacion'].nunique().reset_index()

chart_acto1 = alt.Chart(df_estaciones).mark_bar(color='#4A90E2').encode(
    x=alt.X('anio:O', title='Año'),
    y=alt.Y('estacion:Q', title='Número de estaciones'),
    tooltip=['anio','estacion']
).properties(
    title='Expansión de la red de monitoreo (2000–2026)'
)

chart_acto1.save('visualizacioness/acto1_expansion_red.html')
chart_acto1


alt.Chart(...)

In [11]:
import pandas as pd
import altair as alt
import os

os.makedirs("visualizacioness", exist_ok=True)

df = pd.read_excel("base_analitica_calidad_aire.xlsx")

# Año mínimo en que cada estación aparece
inicio = df.groupby('estacion')['anio'].min().reset_index()
inicio.columns = ['estacion','anio_inicio']

# Contar cuántas estaciones estaban activas cada año
años = pd.DataFrame({'anio': range(2000, 2027)})
años['estaciones_activas'] = años['anio'].apply(lambda a: (inicio['anio_inicio'] <= a).sum())

chart_acto1 = alt.Chart(años).mark_bar(color='#4A90E2').encode(
    x=alt.X('anio:O', title='Año'),
    y=alt.Y('estaciones_activas:Q', title='Número de estaciones activas'),
    tooltip=['anio','estaciones_activas']
).properties(
    title='Expansión de la red de monitoreo (2000–2026)'
)

chart_acto1.save('visualizacioness/acto1_expansion_red.html')
chart_acto1


alt.Chart(...)

In [12]:
import pandas as pd
import altair as alt
import os

os.makedirs("visualizacioness", exist_ok=True)

df = pd.read_excel("base_analitica_calidad_aire.xlsx")

inicio = df.groupby('estacion')['anio'].min().reset_index()
inicio.columns = ['estacion', 'anio_inicio']

años = pd.DataFrame({'anio': range(2000, 2027)})
años['estaciones_activas'] = años['anio'].apply(lambda a: (inicio['anio_inicio'] <= a).sum())

chart_acto1 = alt.Chart(años).mark_bar(color='#4A90E2').encode(
    x=alt.X('anio:O', title='Año'),
    y=alt.Y('estaciones_activas:Q', title='Número de estaciones activas'),
    tooltip=['anio', 'estaciones_activas']
).properties(
    title='Expansión de la red de monitoreo (2000–2026)'
)

chart_acto1.save('visualizacioness/acto1_expansion_red.html')
chart_acto1


alt.Chart(...)

In [14]:
from google.colab import files

uploaded = files.upload()

Saving BASE DE DATOS Cánepa.xlsx to BASE DE DATOS Cánepa.xlsx


In [16]:
from google.colab import files

uploaded = files.upload()

Saving antofagasta.csv to antofagasta.csv
Saving atacama.csv to atacama.csv
Saving bio bio2.csv to bio bio2.csv
Saving coquimbo.csv to coquimbo.csv
Saving la araucania.csv to la araucania.csv
Saving los rios.csv to los rios.csv
Saving maule.csv to maule.csv
Saving nuble.csv to nuble.csv
Saving ohiggins.csv to ohiggins.csv
Saving RM1.csv to RM1.csv
Saving valparaiso.csv to valparaiso.csv


In [17]:
from google.colab import files

uploaded = files.upload()

Saving Tarapacá.csv to Tarapacá.csv


In [18]:
from google.colab import files

uploaded = files.upload()

Saving Arica y Parinacota.csv to Arica y Parinacota.csv


In [19]:
import pandas as pd
import altair as alt
import os
import glob

os.makedirs("visualizacioness", exist_ok=True)

# Leer todos los CSV subidos
archivos = glob.glob("*.csv")

dataframes = []
for archivo in archivos:
    df = pd.read_csv(archivo, sep=";", encoding="utf-8", engine="python")
    # Normalizar columnas: algunas tienen formato ancho (años como columnas)
    if "FECHA (YYMMDD)" in df.columns:
        # Caso Arica/Tarapacá: convertir fecha a año
        df["anio"] = df["FECHA (YYMMDD)"].astype(str).str[:2].astype(int) + 2000
        df = df[["anio","Registros validados"]].rename(columns={"Registros validados":"valor"})
        df["estacion"] = archivo.replace(".csv","")
    else:
        # Caso formato ancho (Antofagasta, Atacama, etc.)
        df = df.melt(id_vars=[df.columns[0]], var_name="anio", value_name="valor")
        df.rename(columns={df.columns[0]:"estacion"}, inplace=True)
        df["anio"] = pd.to_numeric(df["anio"], errors="coerce")
    dataframes.append(df)

df_all = pd.concat(dataframes, ignore_index=True)

# Limpiar valores
df_all["valor"] = pd.to_numeric(df_all["valor"].astype(str).str.replace(",","."),
                                errors="coerce")

# Año real de inicio: primer año con dato válido
inicio = df_all.dropna(subset=["valor"]).groupby("estacion")["anio"].min().reset_index()
inicio.columns = ["estacion","anio_inicio"]

# Contar cuántas estaciones estaban activas cada año
años = pd.DataFrame({"anio": range(2000, 2027)})
años["estaciones_activas"] = años["anio"].apply(lambda a: (inicio["anio_inicio"] <= a).sum())

# Gráfico de expansión de la red
chart_acto1 = alt.Chart(años).mark_bar(color="#4A90E2").encode(
    x=alt.X("anio:O", title="Año"),
    y=alt.Y("estaciones_activas:Q", title="Número de estaciones activas"),
    tooltip=["anio","estaciones_activas"]
).properties(
    title="Expansión de la red de monitoreo (2000–2026)"
)

chart_acto1.save("visualizacioness/acto1_expansion_red.html")
chart_acto1


alt.Chart(...)

In [20]:
import pandas as pd
import altair as alt
import os
import glob

os.makedirs("visualizacioness", exist_ok=True)

# Leer todos los CSV subidos
archivos = glob.glob("*.csv")

dataframes = []
for archivo in archivos:
    # Nombre de la región a partir del archivo
    region = archivo.replace(".csv","")
    df = pd.read_csv(archivo, sep=";", encoding="utf-8", engine="python")

    if "FECHA (YYMMDD)" in df.columns:
        # Caso Arica/Tarapacá: convertir fecha a año
        df["anio"] = df["FECHA (YYMMDD)"].astype(str).str[:2].astype(int) + 2000
        df = df[["anio","Registros validados"]].rename(columns={"Registros validados":"valor"})
        df["estacion"] = region
    else:
        # Caso formato ancho (Antofagasta, Atacama, etc.)
        df = pd.read_csv(archivo, sep=",", encoding="utf-8", engine="python")
        df = df.melt(id_vars=[df.columns[0]], var_name="anio", value_name="valor")
        df.rename(columns={df.columns[0]:"estacion"}, inplace=True)
        df["anio"] = pd.to_numeric(df["anio"], errors="coerce")
    df["region"] = region
    dataframes.append(df)

df_all = pd.concat(dataframes, ignore_index=True)

# Limpiar valores
df_all["valor"] = pd.to_numeric(df_all["valor"].astype(str).str.replace(",","."),
                                errors="coerce")

# Año real de inicio: primer año con dato válido
inicio = df_all.dropna(subset=["valor"]).groupby(["region","estacion"])["anio"].min().reset_index()

# Contar cuántas estaciones estaban activas cada año por región
años = pd.DataFrame({"anio": range(2000, 2027)})
resultados = []
for region in inicio["region"].unique():
    estaciones_region = inicio[inicio["region"] == region]
    for a in años["anio"]:
        activos = (estaciones_region["anio"] <= a).sum()
        resultados.append([region,a,activos])

df_regiones = pd.DataFrame(resultados, columns=["region","anio","estaciones_activas"])

# Gráfico de expansión por región
chart_regiones = alt.Chart(df_regiones).mark_line(point=True).encode(
    x=alt.X("anio:O", title="Año"),
    y=alt.Y("estaciones_activas:Q", title="Número de estaciones activas"),
    color=alt.Color("region:N", title="Región"),
    tooltip=["region","anio","estaciones_activas"]
).properties(
    title="Expansión de la red de monitoreo por región (2000–2026)"
)

chart_regiones.save("visualizacioness/acto1_expansion_por_region.html")
chart_regiones


ParserError: Expected 9 fields in line 3, saw 20

In [21]:
import pandas as pd
import altair as alt
import os
import glob

os.makedirs("visualizacioness", exist_ok=True)

# Leer todos los CSV subidos
archivos = glob.glob("*.csv")

dataframes = []
for archivo in archivos:
    region = archivo.replace(".csv","")
    try:
        # Intentar con ;
        df = pd.read_csv(archivo, sep=";", encoding="utf-8", engine="python")
    except:
        # Si falla, usar ,
        df = pd.read_csv(archivo, sep=",", encoding="utf-8", engine="python")

    # Caso formato con FECHA (Arica/Tarapacá)
    if "FECHA (YYMMDD)" in df.columns:
        df["anio"] = df["FECHA (YYMMDD)"].astype(str).str[:2].astype(int) + 2000
        df = df[["anio","Registros validados"]].rename(columns={"Registros validados":"valor"})
        df["estacion"] = region
    else:
        # Caso formato ancho (años como columnas)
        df = df.melt(id_vars=[df.columns[0]], var_name="anio", value_name="valor")
        df.rename(columns={df.columns[0]:"estacion"}, inplace=True)
        df["anio"] = pd.to_numeric(df["anio"], errors="coerce")
    df["region"] = region
    dataframes.append(df)

df_all = pd.concat(dataframes, ignore_index=True)

# Limpiar valores
df_all["valor"] = pd.to_numeric(df_all["valor"].astype(str).str.replace(",","."),
                                errors="coerce")

# Año real de inicio: primer año con dato válido
inicio = df_all.dropna(subset=["valor"]).groupby(["region","estacion"])["anio"].min().reset_index()

# Contar cuántas estaciones activas por región y año
años = pd.DataFrame({"anio": range(2000, 2027)})
resultados = []
for region in inicio["region"].unique():
    estaciones_region = inicio[inicio["region"] == region]
    for a in años["anio"]:
        activos = (estaciones_region["anio"] <= a).sum()
        resultados.append([region,a,activos])

df_regiones = pd.DataFrame(resultados, columns=["region","anio","estaciones_activas"])

# Gráfico de expansión por región
chart_regiones = alt.Chart(df_regiones).mark_line(point=True).encode(
    x=alt.X("anio:O", title="Año"),
    y=alt.Y("estaciones_activas:Q", title="Número de estaciones activas"),
    color=alt.Color("region:N", title="Región"),
    tooltip=["region","anio","estaciones_activas"]
).properties(
    title="Expansión de la red de monitoreo por región (2000–2026)"
)

chart_regiones.save("visualizacioness/acto1_expansion_por_region.html")
chart_regiones


alt.Chart(...)

In [22]:
import pandas as pd
import altair as alt
import os

os.makedirs("visualizacioness", exist_ok=True)

# Datos simplificados: promedio anual PM2.5 por región
data = {
    "anio": list(range(2000, 2027)),
    "Metropolitana": [
        28, 27, 26, 25, 24, 26, 27, 28, 25, 24, 23, 22, 25, 24, 23, 22, 25, 26, 24, 23, 25, 24, 23, 22, 21, 22, 23
    ],
    "Araucania": [
        30, 32, 31, 33, 34, 35, 36, 37, 38, 36, 33, 34, 50.8, 33, 32, 31, 30, 29, 28, 27, 26, 25, 24, 23, 22, 23, 24
    ]
}

df = pd.DataFrame(data)
df_long = df.melt(id_vars="anio", var_name="region", value_name="pm25")

# Gráfico comparativo
chart = alt.Chart(df_long).mark_line(point=True).encode(
    x=alt.X("anio:O", title="Año"),
    y=alt.Y("pm25:Q", title="PM2.5 promedio anual (µg/m³)"),
    color=alt.Color("region:N", title="Región"),
    tooltip=["region","anio","pm25"]
).properties(
    title="Comparación Sur vs Centro (2000–2026)"
)

chart.save("visualizacioness/acto2_sur_vs_centro.html")
chart


alt.Chart(...)

In [23]:
import pandas as pd
import altair as alt
import os

os.makedirs("visualizacioness", exist_ok=True)

# Datos de ejemplo: pendientes promedio por región (simplificados)
data = {
    "anio": list(range(2000, 2027)),
    "Metropolitana": [
        35, 34, 33, 32, 31, 30, 29, 28, 27, 26, 25, 25, 24, 24, 23, 23, 22, 22, 22, 21, 21, 21, 20, 20, 20, 20, 19
    ],
    "Bio Bio": [
        28, 29, 30, 31, 32, 33, 34, 34, 35, 36, 36, 35, 34, 34, 33, 33, 32, 32, 31, 31, 30, 30, 30, 29, 29, 29, 28
    ],
    "Araucania": [
        30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 50.8, 39, 38, 37, 36, 35, 34, 33, 32, 31, 30, 29, 28, 28, 27
    ]
}

df = pd.DataFrame(data)
df_long = df.melt(id_vars="anio", var_name="region", value_name="pm25")

# Gráfico de tendencias divergentes
chart = alt.Chart(df_long).mark_line(point=True).encode(
    x=alt.X("anio:O", title="Año"),
    y=alt.Y("pm25:Q", title="PM2.5 promedio anual (µg/m³)"),
    color=alt.Color("region:N", title="Región"),
    tooltip=["region","anio","pm25"]
).properties(
    title="Tendencias divergentes de PM2.5 por región (2000–2026)"
)

chart.save("visualizacioness/acto3_tendencias_divergentes.html")
chart


alt.Chart(...)